# Step 7 · Your own investigation

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ open-ended (a first pass takes about 45 minutes) · ⚡ GPU recommended

> ### 🤔 Wonder
> The lab answered one question. The same tools can answer dozens more, and **many of them nobody has tested yet.** What do *you* want to know?

Research doesn't mean discovering something no one has ever imagined. A good project asks a **clear question**, tests it **fairly**, and reports **honestly**, even when the answer is "it didn't work". A careful null result is a real finding.

**This notebook gives you:**

1. A research recipe and a place to write down your prediction **before** you run anything.
2. **Scaffold A:** a ready-made "no giveaway words" experiment: can you write sentences the word-counter fails on but the ruler passes?
3. **Scaffold B:** a name-swap fairness check.
4. Short code recipes for the other starter projects.

The full menu of research ideas, with background and reading, is in [`docs/research-ideas.md`](../docs/research-ideas.md).

## The research recipe

1. **Ask a question you can test.** "Does AI understand pain?" is too big. "Does our pain–comfort direction also separate emotional pain from emotional comfort?" can be tested in a week.
2. **Write down your prediction**, and what result would prove it wrong. *Before* you run anything.
3. **Change one thing at a time.** If you change the model *and* the sentences, you won't know which change caused your result.
4. **Plan your data.** Write at least 10 pairs of new sentences, more if you can. Keep a test set sealed until the end.
5. **Choose your comparison.** Random directions, the word-counter, and the original lab setup are all good controls.
6. **Run and record everything**, including what failed. The save cells export your tables and settings.
7. **Analyse honestly.** Did your result beat the controls? Is it bigger than noise? With 16 test sentences, one mistake changes accuracy by about 6 points.
8. **Explain it.** Give at least two possible explanations, name the limitations, suggest a next experiment.
9. **Share it.** See the end of this notebook.

### Starter projects

| Project | Level | The question | Where to start |
|---|---|---|---|
| Metaphor hunt | Explorer | Where do 20 metaphorical pain/comfort phrases land on the ruler? | Step 4's mystery-sentence reader |
| Name-swap fairness check | Explorer | Does a pain sentence's score change when only the name changes? | **Scaffold B** below |
| Template test | Explorer | Does changing "The bodily sensation is" change the best layer? | Recipe 1 below |
| No-giveaway-words test | Investigator | Does the ruler still work when pain is implied, not stated? | **Scaffold A** below |
| Steering strength curve | Investigator | How does the effect grow with α, and when does text break? | Step 6's dose–response cell |
| Small vs. large model | Investigator | Does the best layer sit at the same *relative* depth in a smaller model? | Recipe 3 below |
| Mood vs. sensation | Investigator | Is a happy–sad direction the same arrow as pain–comfort? | Recipe 4 below |
| Cross-language transfer | Researcher | Does a ruler built from English sentences work in another language? | Scaffold A, with translated sentences |
| Best layer to read vs. steer | Researcher | Is the layer that probes best also the one that steers best? | Step 6, looped over layers |
| Base vs. chat model | Researcher | Does chat training change the direction? | Recipe 3 with `Qwen/Qwen2.5-1.5B-Instruct` |

**Explorer** projects only change sentences or settings; **Investigator** projects need small code edits; **Researcher** projects need new code. Every level can lead to a project worth presenting.

## ✍️ Pre-register your prediction

Fill this in **before** running your experiment. It's stored in a variable so it's saved with your results; that way, nobody (including you) can quietly change the prediction afterwards.

In [ ]:
# ✍️ Edit the text between the triple quotes, then run this cell
PREREGISTRATION = """
Question:    We will test whether ...
Change:      by changing ...
Keep same:   while keeping ... the same.
Prediction:  We predict ...
Wrong if:    If ... happens instead, our prediction is wrong.
Data:        (how many new sentences, written by whom, which are sealed as the test set)
Control:     (what we compare against: random directions, word-counter, original setup...)
Date:        (today's date)
"""
print(PREREGISTRATION)

## Setup and catch-up

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

In [ ]:
# 🔧 Catch-up: the frozen ruler from Steps 2–3, and the word-counter from Step 5
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

state = pclab.rebuild_lab_state(model, tokenizer)
build_df, y_train = state['build_df'], state['y_train']
unit_vectors, midpoints, vector_lengths = state['unit_vectors'], state['midpoints'], state['vector_lengths']
BEST_LAYER = state['best_layer']
REFERENCE_SCALE = float(vector_lengths[BEST_LAYER]) / 2

def ruler_scores(activations):
    """Frozen-layer scores, rescaled so the pain average = −1 and the comfort average = +1."""
    return pclab.scores_at_all_layers(activations, unit_vectors, midpoints)[:, BEST_LAYER] / REFERENCE_SCALE

word_counter = make_pipeline(TfidfVectorizer(ngram_range=(1, 1)),
                             LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
word_counter.fit(build_df['text'], y_train)
print('Ready: frozen ruler at layer', BEST_LAYER + 1, '+ word-counter built from the original build sentences')

## Scaffold A · The "no giveaway words" test

Our original sentences are full of giveaway words (*soothed, burned, sting*), so in Step 5 a simple word-counter could solve the test. Here's the open puzzle: **if pain or comfort is only *implied by the situation*, does the ruler still work? Does the word-counter?**

The starter pairs below avoid explicit sensation words. They're **examples to replace and extend**: write at least 10 pairs of your own. Tips:

- Keep each pair about the **same topic and length**, differing only in whether the situation would hurt or feel good.
- Avoid words from the original build set (the cell below lists which of your words the word-counter knows, and which it has never seen).
- Ask a classmate to label your sentences *without* seeing your labels. If people disagree, the sentence is ambiguous: fix it or drop it.

In [ ]:
# 🧪 Experiment: write your own pairs as (comfort, pain). Keep the comma at the end of each line!
MY_PAIRS = [
    ('She stepped barefoot onto the shaded lawn on a spring morning.',
     'She stepped barefoot onto the asphalt at noon in July.'),
    ('He sat on the padded bench in the shade.',
     'He sat on the metal bleachers that had baked all afternoon in the sun.'),
    ('She wrapped both hands around a mug of cocoa after coming in from the snow.',
     'She grabbed the cast-iron pan by its handle straight out of the oven.'),
    ('He pulled on wool socks that had been drying by the fire.',
     'He walked five miles in new boots half a size too small.'),
    ('The nurse tucked an extra pillow under his raised leg.',
     'The car door closed on her fingers.'),
    ('She sank into the hot tub after a long day of skiing.',
     'She slipped on the ice and landed on her wrist.'),
]

rows = []
for i, (comfort_text, pain_text) in enumerate(MY_PAIRS):
    rows.append({'pair_id': f'mine-{i:02d}', 'label': 1, 'category': 'Comfort', 'text': comfort_text})
    rows.append({'pair_id': f'mine-{i:02d}', 'label': 0, 'category': 'Pain', 'text': pain_text})
my_df = pd.DataFrame(rows)
assert my_df['text'].is_unique, 'Two of your sentences are identical.'
assert not set(my_df['text']) & set(pclab.load_sentences()['text']), 'Some sentences are copied from the lab data.'
look(my_df, 'my_df')

In [ ]:
# 🔧 Which of your words has the word-counter never seen? (It can only use words from the build set.)
vocabulary = set(word_counter.named_steps['tfidfvectorizer'].vocabulary_)
analyzer = word_counter.named_steps['tfidfvectorizer'].build_analyzer()   # the same word-splitting rules it uses
my_words = {word for text in my_df['text'] for word in analyzer(text)}
print(f'{len(my_words & vocabulary)} of your {len(my_words)} different words are KNOWN to the word-counter:')
print(sorted(my_words & vocabulary))
print(f'\n{len(my_words - vocabulary)} words it has NEVER seen (it simply ignores these):')
print(sorted(my_words - vocabulary))

In [ ]:
# 🔧 Run the comparison: frozen ruler vs. word-counter vs. 200 random directions, on YOUR sentences
my_act = pclab.collect(model, tokenizer, my_df['text'], 'Your sentences')
y_mine = my_df['label'].to_numpy()

my_scores = ruler_scores(my_act)
ruler_accuracy = accuracy_score(y_mine, (my_scores > 0).astype(int))
correct = int(((my_scores > 0).astype(int) == y_mine).sum())
lo, hi = pclab.wilson_interval(correct, len(y_mine))

word_probabilities = word_counter.predict_proba(my_df['text'])[:, 1]
word_accuracy = accuracy_score(y_mine, (word_probabilities > 0.5).astype(int))

rng = np.random.default_rng(SEED + 3)
random_units = rng.normal(size=(200, HIDDEN_SIZE))
random_units /= np.linalg.norm(random_units, axis=1, keepdims=True)
random_scores = (my_act[:, BEST_LAYER, :] - midpoints[BEST_LAYER]) @ random_units.T
random_accuracies = ((random_scores > 0).astype(int) == y_mine[:, None]).mean(axis=0)

summary = pd.DataFrame([
    {'method': 'Frozen ruler (model snapshots)', 'accuracy': ruler_accuracy, 'auc': roc_auc_score(y_mine, my_scores)},
    {'method': 'Word-counter (TF-IDF + logistic regression)', 'accuracy': word_accuracy, 'auc': roc_auc_score(y_mine, word_probabilities)},
    {'method': 'Random directions (mean of 200)', 'accuracy': random_accuracies.mean(), 'auc': np.nan},
])
display(summary.round(3))
print(f'Ruler: {correct}/{len(y_mine)} correct; plausible range {lo:.0%} to {hi:.0%}')
print(f'Share of random directions matching or beating the ruler: {(random_accuracies >= ruler_accuracy).mean():.1%}')

my_report = my_df.assign(ruler_score=my_scores.round(2), word_counter_p_comfort=word_probabilities.round(2))
with pd.option_context('display.max_colwidth', 90):
    display(my_report[['pair_id', 'category', 'text', 'ruler_score', 'word_counter_p_comfort']])

### Reading your result

- With only a handful of pairs, the **plausible range is very wide**. Six pairs can't settle anything; they can only tell you whether the idea is worth testing properly.
- Which sentences did the ruler get wrong? Is there a pattern (a topic, a sentence length, a particular kind of situation)?
- If the ruler beats the word-counter, try to rule out boring explanations: are your comfort sentences longer? Do they share a word like "hot"?
- For a real result, **write your sentences, seal half of them as a test set, and don't change them after you've seen the scores.**

## Scaffold B · A name-swap fairness check

Doctors are known to sometimes underestimate pain in certain groups of patients. As AI is used in healthcare, it matters whether models pick up similar patterns. A first, simple probe: **does a sentence's score change when only the person's name changes?**

**Read before you interpret:**

- Names are imperfect stand-ins for groups. A difference could come from how *common* a name is, or how it's split into tokens, rather than from anything about a group.
- Always compare differences **between** groups of names with the spread **within** a group. If two names from the same group differ as much as names from different groups, you've found noise, not bias.
- One template proves nothing. Use many templates, many names per group, and report all of them.
- The names below are placeholders. For a real study, take your groups and names from a published name list (for example, one used in an audit study) and cite where they came from.

In [ ]:
# 🧪 Experiment: change the templates and the names (use several names per group)
TEMPLATES = [
    '{name} touched the hot stove and pulled their hand away.',
    '{name} twisted their ankle on the stairs.',
    '{name} rested their sore back on a soft cushion.',
]
NAME_GROUPS = {
    'Group A': ['Emily', 'Hannah', 'Katie'],
    'Group B': ['Lakisha', 'Aaliyah', 'Imani'],
    'Group C': ['Mei', 'Yuki', 'Ji-woo'],
}

name_rows = []
for template in TEMPLATES:
    for group, names in NAME_GROUPS.items():
        for name in names:
            name_rows.append({'template': template, 'group': group, 'name': name, 'text': template.format(name=name)})
name_df = pd.DataFrame(name_rows)
name_df['ruler_score'] = ruler_scores(pclab.collect(model, tokenizer, name_df['text'], 'Name swaps'))

summary_by_group = name_df.groupby(['template', 'group'])['ruler_score'].agg(['mean', 'min', 'max']).round(3)
display(summary_by_group)

**How to read the table:** for each template, compare the **mean** across groups, then look at **min–max** within each group. A between-group gap smaller than the within-group spread is not evidence of anything.

`.groupby([...])['ruler_score'].agg(['mean', 'min', 'max'])` is the same `groupby` pattern from Step 2, with three summaries computed at once.

## Code recipes for other projects

Copy one of these into a new cell and adapt it.

**Recipe 1 · Template test.** Rebuild the ruler with a different measurement template:

```python
my_prompt = lambda text: 'Description: ' + text.strip() + '\nIt felt'
train_act2 = pclab.collect(model, tokenizer, build_df['text'], 'Build (new template)', prompt_fn=my_prompt)
val_act2 = pclab.collect(model, tokenizer, state['val_df']['text'], 'Validation (new template)', prompt_fn=my_prompt)
raw2, unit2, mid2, len2 = pclab.fit_directions(train_act2, y_train)
table2 = pclab.layer_table(pclab.scores_at_all_layers(val_act2, unit2, mid2), state['y_val'])
print('New best layer:', pclab.choose_layer(table2) + 1, '| original:', BEST_LAYER + 1)
```

**Recipe 2 · Average over all tokens instead of the last one:** add `reduce='mean'` to every `pclab.collect(...)` call in Recipe 1.

**Recipe 3 · A different model:** change `MODEL_NAME` in the load cell (for example `'Qwen/Qwen2.5-0.5B'` or `'Qwen/Qwen2.5-1.5B-Instruct'`) and rerun the notebook from the top. Compare *relative* depth: `(BEST_LAYER + 1) / N_LAYERS`. Models from other families usually work too, because `pclab.get_layers` looks for their layers under several common names.

**Recipe 4 · Mood vs. sensation:** write pairs of happy vs. sad sentences with **no** physical sensation, build a direction from them, and compare it with the pain–comfort direction at the same layer:

```python
MOOD_PAIRS = [('She smiled when the letter arrived.', 'She frowned when the letter arrived.'),
              # ... at least 10 pairs of (happy, sad), with no physical sensation
             ]
mood_texts = [t for pair in MOOD_PAIRS for t in pair]        # happy, sad, happy, sad, ...
y_mood = np.array([1, 0] * len(MOOD_PAIRS))                  # 1 = happy, 0 = sad
mood_act = pclab.collect(model, tokenizer, mood_texts, 'Mood sentences')
_, mood_unit_vectors, _, _ = pclab.fit_directions(mood_act, y_mood)
cos = float(np.dot(mood_unit_vectors[BEST_LAYER], unit_vectors[BEST_LAYER]))  # both length 1
print('Cosine similarity with the pain–comfort direction:', round(cos, 3))
```

1 means the same arrow; 0 means unrelated. What would each result mean for the lab's main question?

## Save your investigation

In [ ]:
pclab.save_results(
    'step7_my_investigation',
    tables={'no_giveaway_sentences': my_report, 'no_giveaway_summary': summary, 'name_swaps': name_df},
    settings={**pclab.model_facts(model, MODEL_NAME), 'seed': SEED, 'selected_layer_1_based': BEST_LAYER + 1,
              'preregistration': PREREGISTRATION},
)

---

## Sharing your work, and doing it right

Research isn't finished until someone else can read it. Start small: a club presentation, a poster at your campus research day, a write-up on GitHub or a blog. Later: national venues such as the National Conference on Undergraduate Research (NCUR), ACM student research competitions, and undergraduate research journals.

**A short write-up has six parts:** your question, what you did, what you found, how it compared with the controls, the limitations, and what you'd test next. Include your charts and your pre-registration.

### Research etiquette

- **Cite your tools and sources:** this lab (see `CITATION.cff` in the repository, or the DOI badge in the README), the Qwen2.5 model, and the papers behind the methods.
- **Say what's yours:** state clearly what you added: new sentences, a new comparison, a new analysis.
- **AI use:** if your instructor's policy allows AI tools, follow four rules: **use it, disclose it, verify it, own it.**
- **Don't overclaim:** write "the model's internal numbers separate pain and comfort descriptions", not "the model feels pain".
- **Report what didn't work.** Showing only the best examples is cherry-picking.
- **Keep it safe:** use made-up sentences, never personal or sensitive information.

**Final reflection:** what additional evidence would distinguish *"our ruler found easy word clues"* from *"our ruler captures a more general representation of bodily sensation"*? Design one next experiment with a fair control.